In [1]:
import neo4j
import math
import csv
from geographiclib.geodesic import Geodesic
import math
import numpy as np
import pandas as pd

import psycopg2

In [2]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

In [3]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [4]:
cursor = connection.cursor()

In [5]:
def my_read_csv_file(file_name, limit):
    "read the csv file and print only the first limit rows"
    
    csv_file = open(file_name, "r")
    
    csv_data = csv.reader(csv_file)
    
    i = 0
    
    for row in csv_data:
        i += 1
        if i <= limit:
            print(row)
            
    print("\nPrinted ", min(limit, i), "lines of ", i, "total lines.")

In [6]:
connection.rollback()

query = """

drop table if exists stations;

"""

cursor.execute(query)

connection.commit()


In [7]:
connection.rollback()

query = """

drop table if exists lines;

"""

cursor.execute(query)

connection.commit()


In [8]:
connection.rollback()

query = """

drop table if exists distances;

"""

cursor.execute(query)

connection.commit()


In [9]:
connection.rollback()

query = """

create table stations (
  station varchar(64),
  latitude numeric(9,6),
  longitude numeric(9,6),
  traffic_17 numeric(10),
  primary key (station)
);

"""

cursor.execute(query)

connection.commit()

In [10]:
connection.rollback()

query = """

create table lines (
  line varchar(6),
  sequence numeric(2),
  station varchar(64),
  primary key (line, sequence)
);

"""

cursor.execute(query)

connection.commit()

In [11]:
connection.rollback()

query = """

create table distances (
  station_1 varchar(64),
  station_2 varchar(64),
  distance_meters numeric(10),
  primary key (station_1, station_2)
);

"""

cursor.execute(query)

connection.commit()

In [12]:
connection.rollback()

query = """

copy stations
from '/user/projects/project-3-group4/code/stations.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [13]:
connection.rollback()

query = """

copy lines
from '/user/projects/project-3-group4/code/lines.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [14]:
connection.rollback()

query = """

copy distances
from '/user/projects/project-3-group4/code/distances.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [15]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stations
order by station

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station,latitude,longitude,traffic_17
0,103 St,40.796093,-73.961460,52
1,103 St-Corona Plaza,40.749866,-73.862700,60
2,104 St,40.688445,-73.841007,7
3,110 St,40.795020,-73.944250,46
4,110 St-Malcolm X Plaza,40.802097,-73.949620,45
...,...,...,...,...
358,Woodhaven Blvd,40.713493,-73.860405,46
359,Woodlawn,40.886036,-73.878750,21
360,World Trade Center,40.702150,-73.801110,106
361,WTC Cortlandt,40.711834,-74.012190,157


In [16]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from lines
order by line, sequence

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,line,sequence,station
0,1,1,Van Cortlandt Park–242 St
1,1,2,238 St
2,1,3,231 St
3,1,4,Marble Hill–225 St
4,1,5,215 St
...,...,...,...
622,Z,17,Bowery
623,Z,18,Canal St
624,Z,19,Chambers St
625,Z,20,Fulton St


In [17]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from distances
order by station_1, station_2

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station_1,station_2,distance_meters
0,103 St,96 St,946
1,103 St-Corona Plaza,Junction Blvd,581
2,104 St,111 St,6356
3,104 St,Woodhaven Blvd,628
4,110 St,103 St,2090
...,...,...,...
482,Woodhaven Blvd,85 St–Forest Pkwy,729
483,Woodhaven Blvd,Grand Av,5261
484,Woodlawn,Mosholu Pkwy,858
485,WTC Cortlandt,Rector St,499


In [18]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select station
from stations
order by station

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station
0,103 St
1,103 St-Corona Plaza
2,104 St
3,110 St
4,110 St-Malcolm X Plaza
...,...
358,Woodhaven Blvd
359,Woodlawn
360,World Trade Center
361,WTC Cortlandt


In [19]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [20]:
session = driver.session(database="neo4j")

In [21]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [22]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [23]:
def my_neo4j_number_nodes_relationships():
    "print the number of nodes and relationships"
   
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    print("-------------------------")
    print("  Nodes:", number_nodes)
    print("  Relationships:", number_relationships)
    print("-------------------------")


In [24]:
def my_neo4j_create_node(station_name):
    "create a node with label Station"
    
    query = """
    
    CREATE (:Station {name: $station_name})
    
    """
    
    session.run(query, station_name=station_name)
    

In [25]:
def my_neo4j_create_relationship_one_way(from_station, to_station, weight):
    "create a relationship one way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [26]:
def my_neo4j_create_relationship_two_way(from_station, to_station, weight):
    "create relationships two way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to),
           (to)-[:LINK {weight: $weight}]->(from)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

# Rerun the next 10 cells to reset the graph's weight

In [57]:
my_neo4j_wipe_out_database()

In [58]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 0
  Relationships: 0
-------------------------


In [59]:
connection.rollback()

query = """

select station
from stations
order by station

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    
    my_neo4j_create_node('depart ' + station)
    my_neo4j_create_node('arrive ' + station)
    

In [60]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 726
  Relationships: 0
-------------------------


In [61]:
connection.rollback()

query = """

select station, line
from lines
order by station, line

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    line = row[1]
    
    depart = 'depart ' + station
    arrive = 'arrive ' + station
    line_station = line + ' ' + station
    
    my_neo4j_create_node(line_station)
    my_neo4j_create_relationship_one_way(depart, line_station, 0)
    my_neo4j_create_relationship_one_way(line_station, arrive, 0)
    

In [62]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 1353
  Relationships: 1258
-------------------------


In [63]:
connection.rollback()

query = """

select a.station, a.line as from_line, b.line as to_line, s.traffic_17
from lines a
     join lines b
       on a.station = b.station and a.line <> b.line 
     join stations s
       on a.station = s.station
order by 1, 2, 3

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    from_line = row[1]
    to_line = row[2]
    transfer_time = int(row[3])
    
    from_station = from_line + ' ' + station
    to_station = to_line + ' ' + station
    
    my_neo4j_create_relationship_one_way(from_station, to_station, transfer_time)
    

In [64]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 1353
  Relationships: 2330
-------------------------


In [65]:
connection.rollback()

query = """

select a.line, a.station as from_station, b.station as to_station, t.distance_meters
from lines a
  join lines b
    on a.line = b.line and b.sequence = (a.sequence + 1)
  join distances t
    on (a.station = t.station_1 and b.station = t.station_2)
        or (a.station = t.station_2 and b.station = t.station_1)
order by line, from_station, to_station

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    line = row[0]
    from_station = line + ' ' + row[1]
    to_station = line + ' ' + row[2]
    travel_time = int(row[3])
    
    my_neo4j_create_relationship_two_way(from_station, to_station, travel_time)
    

In [66]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 1353
  Relationships: 3568
-------------------------


# Shortest Path Algorithm

We took the NYC subway ridership from the beginning of this year to November and calculated the max ridership at each station (usually at 8am-9am). We used that number to estimate the current max capacity of each station and tracked which stations would hit that max capacity if we increased ridership by 20% and 40%.

We considered those stations as "bottlenecked" and multiplied the weights of the adjacent edges by a penalty to simulate how much longer the shortest path will increase by between any two stations if NYC subway ridership increased by 20% and 40%.

In [77]:
def my_neo4j_shortest_path(from_station, to_station):
    "given a from station and to station, run and print the shortest path"
    
    query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
    session.run(query)

    query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
    session.run(query)

    query = """

    MATCH (source:Station {name: $source}), (target:Station {name: $target})
    CALL gds.shortestPath.dijkstra.stream(
        'ds_graph', 
        { sourceNode: source, 
          targetNode: target, 
          relationshipWeightProperty: 'weight'
        }
    )
    YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
    RETURN
        gds.util.asNode(sourceNode).name AS from,
        gds.util.asNode(targetNode).name AS to,
        totalCost,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodes,
        costs
    ORDER BY index

    """

    result = session.run(query, source=from_station, target=to_station)
    
    for r in result:
        
        total_cost = int(r['totalCost'])
        
        print("\n--------------------------------")
        print("   Total Cost: ", total_cost)
        print("--------------------------------")
        
        nodes = r['nodes']
        costs = r['costs']
        
        i = 0
        previous = 0
        
        for n in nodes:
            
            print(n + ", " + str(int(costs[i]) - previous)  + ", " + str(int(costs[i])))
            
            previous = int(costs[i])
            i += 1
    

# Shortest Path Algorithm with no added bottleneck penalty

In [78]:
my_neo4j_shortest_path('depart Van Cortlandt Park–242 St', 'arrive South Ferry')


--------------------------------
   Total Cost:  26886
--------------------------------
depart Van Cortlandt Park–242 St, 0, 0
1 Van Cortlandt Park–242 St, 0, 0
1 238 St, 545, 545
1 231 St, 727, 1272
1 Marble Hill–225 St, 636, 1908
1 215 St, 730, 2638
1 207 St, 614, 3252
1 Dyckman St, 725, 3977
1 191 St, 1156, 5133
1 181 St, 727, 5860
1 168 St, 1228, 7088
1 157 St, 867, 7955
1 145 St, 952, 8907
A 145 St, 47, 8954
A 125 St, 1571, 10525
A 59 St-Columbus Circle, 5365, 15890
C 59 St-Columbus Circle, 521, 16411
C 50 St, 752, 17163
E 50 St, 133, 17296
E 42 St–Port Authority Bus Terminal, 1291, 18587
E 34 St–Penn Station, 290, 18877
E 23 St, 933, 19810
R 23 St, 157, 19967
R 14 St–Union Sq, 621, 20588
R 8 St–NYU, 642, 21230
R Prince St, 792, 22022
R Canal St, 741, 22763
R City Hall, 916, 23679
R Cortlandt St, 916, 24595
R Rector St, 1777, 26372
1 Rector St, 53, 26425
1 South Ferry, 461, 26886
arrive South Ferry, 0, 26886


# Bottleneck stations for 20% increased ridership at 17:00 (140 stations)

In [67]:
bottleneck_stations_120 = [
    "1 Av",
    "103 St",
    "116 St-Columbia University",
    "125 St",
    "135 St",
    "138 St-Grand Concourse",
    "14 St",
    "14 St-Union Sq",
    "149 St-Grand Concourse",
    "161 St-Yankee Stadium",
    "168 St",
    "18 St",
    "2 Av",
    "21 St",
    "23 St",
    "25 St",
    "28 St",
    "3 Av",
    "33 St",
    "34 St-Herald Sq",
    "34 St-Hudson Yards",
    "34 St-Penn Station",
    "36 St",
    "4 Av",
    "47-50 Sts-Rockefeller Ctr",
    "49 St",
    "5 Av/53 St",
    "5 Av/59 St",
    "50 St",
    "55 St",
    "57 St",
    "57 St-7 Av",
    "59 St-Columbus Circle",
    "66 St-Lincoln Center",
    "68 St-Hunter College",
    "7 Av",
    "72 St",
    "77 St",
    "79 St",
    "8 Av",
    "8 St-NYU",
    "81 St-Museum of Natural History",
    "82 St-Jackson Hts",
    "86 St",
    "96 St",
    "Alabama Av",
    "Aqueduct Racetrack",
    "Astor Pl",
    "Atlantic Av-Barclays Ctr",
    "Avenue I",
    "Beach 105 St",
    "Beach 90 St",
    "Beach 98 St",
    "Bedford Av",
    "Bergen St",
    "Bowery",
    "Bowling Green",
    "Broad St",
    "Broadway",
    "Broadway-Lafayette St",
    "Brooklyn Bridge-City Hall",
    "Bryant Pk",
    "Canal St",
    "Cathedral Pkwy (110 St)",
    "Chambers St",
    "Christopher St-Sheridan Sq",
    "City Hall",
    "Clinton-Washington Avs",
    "Coney Island-Stillwell Av",
    "Court Sq",
    "Court St",
    "DeKalb Av",
    "Delancey St",
    "East Broadway",
    "Eastern Pkwy-Brooklyn Museum",
    "Flushing Av",
    "Franklin St",
    "Fulton St",
    "Grand Central-42 St",
    "Grand St",
    "Greenpoint Av",
    "High St",
    "Houston St",
    "Howard Beach-JFK Airport",
    "Hoyt St",
    "Hoyt-Schermerhorn Sts",
    "Jay St-MetroTech",
    "Jefferson St",
    "Lafayette Av",
    "Lexington Av",
    "Lexington Av-53 St",
    "Lexington Av/63 St",
    "Lorimer St",
    "Marcy Av",
    "Morgan Av",
    "Nassau Av",
    "Nevins St",
    "Prince St",
    "Queens Plaza",
    "Queensboro Plaza",
    "RI Tramway (Manhattan)",
    "RI Tramway (Roosevelt)",
    "Rector St",
    "Rockaway Park-Beach 116 St",
    "Smith-9 Sts",
    "South Ferry",
    "Spring St",
    "St George",
    "Times Sq-42 St",
    "Tompkinsville",
    "W 4 St-Wash Sq",
    "W 8 St-NY Aquarium",
    "WTC Cortlandt",
    "Wall St",
    "West Farms Sq-E Tremont Av",
    "Westchester Sq-E Tremont Av",
    "York St"
]

penalty = 1.5

In [68]:
def apply_bottleneck_penalty(stations, multiplier):

    for station in stations:

        # Travel edges touching this station
        query_travel = """
        
        MATCH (a:Station)-[r:LINK]->(b:Station)
        WHERE a.name CONTAINS $station OR b.name CONTAINS $station
        SET r.weight = r.weight * $multiplier
        
        """
        
        session.run(query_travel, station=station, multiplier=multiplier)

        # Transfer edges inside the station
        query_transfer = """
        
        MATCH (a:Station)-[r:LINK]->(b:Station)
        WHERE a.name ENDS WITH $station AND b.name ENDS WITH $station
        SET r.weight = r.weight * $multiplier
        
        """
        
        session.run(query_transfer, station=station, multiplier=multiplier)

        # Arrive/depart edges
        query_arrive_depart = """
        
        MATCH (a:Station)-[r:LINK]->(b:Station)
        WHERE a.name = $depart OR b.name = $arrive
        SET r.weight = r.weight * $multiplier
        
        """
        
        session.run(query_arrive_depart, depart="depart " + station, arrive="arrive " + station, multiplier=multiplier)

# Shortest Path Algorithm with bottleneck penalty of 20% increased ridership

In [69]:
apply_bottleneck_penalty(bottleneck_stations_120, penalty)

In [82]:
my_neo4j_shortest_path('depart Van Cortlandt Park–242 St', 'arrive South Ferry')


--------------------------------
   Total Cost:  48248
--------------------------------
depart Van Cortlandt Park–242 St, 0, 0
1 Van Cortlandt Park–242 St, 0, 0
1 238 St, 545, 545
1 231 St, 727, 1272
1 Marble Hill–225 St, 954, 2226
1 215 St, 1095, 3321
1 207 St, 614, 3935
1 Dyckman St, 725, 4660
1 191 St, 1156, 5816
1 181 St, 727, 6543
A 181 St, 41, 6584
A 175 St, 501, 7085
A 168 St, 1113, 8198
A 145 St, 2728, 10926
1 145 St, 47, 10973
1 137 St–City College, 854, 11827
1 125 St, 1838, 13665
3 125 St, 532, 14197
3 116 St, 2313, 16510
3 110 St-Malcolm X Plaza, 553, 17063
3 96 St, 3174, 20237
3 72 St, 6757, 26994
3 Times Square-42 St, 4059, 31053
3 34 St-Penn Station, 1278, 32331
C 34 St-Penn Station, 992, 33323
C 23 St, 2099, 35422
W 23 St, 353, 35775
W 14 St–Union Sq, 1398, 37173
W 8 St–NYU, 963, 38136
W Prince St, 1188, 39324
W Canal St, 1667, 40991
W City Hall, 2061, 43052
W Cortlandt St, 1374, 44426
W Rector St, 2665, 47091
1 Rector St, 120, 47211
1 South Ferry, 1037, 48248
arrive S

# Bottleneck stations for 40% increased ridership at 17:00 (172 stations)

In [53]:
bottleneck_stations_140 = [
    "1 Av",
    "103 St",
    "116 St",
    "116 St-Columbia University",
    "125 St",
    "135 St",
    "138 St-Grand Concourse",
    "14 St",
    "8 Av",
    "14 St-Union Sq",
    "149 St-Grand Concourse",
    "161 St-Yankee Stadium",
    "168 St",
    "174-175 Sts",
    "18 St",
    "2 Av",
    "21 St",
    "21 St-Queensbridge",
    "215 St",
    "23 St",
    "25 St",
    "28 St",
    "3 Av",
    "3 Av-149 St",
    "33 St",
    "33 St-Rawson St",
    "34 St-Herald Sq",
    "34 St-Hudson Yards",
    "34 St-Penn Station",
    "36 St",
    "39 Av-Dutch Kills",
    "4 Av",
    "47-50 Sts-Rockefeller Ctr",
    "49 St",
    "5 Av/53 St",
    "5 Av/59 St",
    "50 St",
    "55 St",
    "57 St",
    "57 St-7 Av",
    "59 St-Columbus Circle",
    "66 St-Lincoln Center",
    "68 St-Hunter College",
    "7 Av",
    "72 St",
    "77 St",
    "79 St",
    "8 St-NYU",
    "81 St-Museum of Natural History",
    "82 St-Jackson Hts",
    "86 St",
    "96 St",
    "Alabama Av",
    "Aqueduct Racetrack",
    "Astor Pl",
    "Atlantic Av-Barclays Ctr",
    "Avenue I",
    "Beach 105 St",
    "Beach 67 St",
    "Beach 90 St",
    "Beach 98 St",
    "Bedford Av",
    "Bergen St",
    "Bowery",
    "Bowling Green",
    "Brighton Beach",
    "Broad St",
    "Broadway",
    "Broadway-Lafayette St",
    "Brooklyn Bridge-City Hall",
    "Bryant Pk",
    "Canal St",
    "Cathedral Pkwy (110 St)",
    "Chambers St",
    "Christopher St-Sheridan Sq",
    "City Hall",
    "Clark St",
    "Clinton-Washington Avs",
    "Coney Island-Stillwell Av",
    "Court Sq",
    "Court St",
    "Cypress Av",
    "DeKalb Av",
    "Delancey St",
    "East Broadway",
    "Eastern Pkwy-Brooklyn Museum",
    "Flushing Av",
    "Flushing-Main St",
    "Fort Hamilton Pkwy",
    "Franklin Av",
    "Franklin St",
    "Fulton St",
    "Grand Central-42 St",
    "Grand St",
    "Greenpoint Av",
    "Hewes St",
    "High St",
    "Houston St",
    "Howard Beach-JFK Airport",
    "Hoyt St",
    "Hoyt-Schermerhorn Sts",
    "Hunters Point Av",
    "Jay St-MetroTech",
    "Jefferson St",
    "Lafayette Av",
    "Lexington Av",
    "Lexington Av-53 St",
    "Lexington Av/63 St",
    "Longwood Av",
    "Lorimer St",
    "Marcy Av",
    "Morgan Av",
    "Nassau Av",
    "Nevins St",
    "Ocean Pkwy",
    "Park Pl",
    "Prince St",
    "Queens Plaza",
    "Queensboro Plaza",
    "RI Tramway (Manhattan)",
    "RI Tramway (Roosevelt)",
    "Rector St",
    "Rockaway Park-Beach 116 St",
    "Simpson St",
    "Smith-9 Sts",
    "South Ferry",
    "Spring St",
    "St George",
    "Sutphin Blvd-Archer Av-JFK Airport",
    "Times Sq-42 St",
    "Tompkinsville",
    "Union St",
    "W 4 St-Wash Sq",
    "W 8 St-NY Aquarium",
    "WTC Cortlandt",
    "Wall St",
    "West Farms Sq-E Tremont Av",
    "Westchester Sq-E Tremont Av",
    "Winthrop St",
    "York St"
]

penalty = 1.5

# Shortest Path Algorithm with bottleneck penalty of 40% increased ridership

In [54]:
apply_bottleneck_penalty(bottleneck_stations_140, penalty)

In [96]:
my_neo4j_shortest_path('depart Van Cortlandt Park–242 St', 'arrive South Ferry')


--------------------------------
   Total Cost:  50535
--------------------------------
depart Van Cortlandt Park–242 St, 0, 0
1 Van Cortlandt Park–242 St, 0, 0
1 238 St, 545, 545
1 231 St, 727, 1272
1 Marble Hill–225 St, 954, 2226
1 215 St, 1642, 3868
1 207 St, 921, 4789
1 Dyckman St, 725, 5514
1 191 St, 1156, 6670
1 181 St, 727, 7397
A 181 St, 41, 7438
A 175 St, 501, 7939
A 168 St, 1113, 9052
A 145 St, 2729, 11781
1 145 St, 47, 11828
1 137 St–City College, 854, 12682
1 125 St, 1838, 14520
3 125 St, 531, 15051
3 116 St, 3470, 18521
3 110 St-Malcolm X Plaza, 829, 19350
3 96 St, 3174, 22524
3 72 St, 6757, 29281
3 Times Square-42 St, 4059, 33340
3 34 St-Penn Station, 1278, 34618
C 34 St-Penn Station, 992, 35610
C 23 St, 2100, 37710
R 23 St, 353, 38063
R 14 St–Union Sq, 1397, 39460
R 8 St–NYU, 963, 40423
R Prince St, 1188, 41611
R Canal St, 1667, 43278
R City Hall, 2061, 45339
R Cortlandt St, 1374, 46713
R Rector St, 2666, 49379
1 Rector St, 119, 49498
1 South Ferry, 1037, 50535
arrive S

# Betweenness Centrality (Network Resilience)

We use Neo4j Graph Data Science (GDS) to compute betweenness centrality on the subway graph.
This measures how often a node lies on shortest paths between all pairs of nodes, using the
`weight` property as travel cost. High betweenness means the station is a critical "bridge"
in the network whose congestion or failure would strongly affect many trips.

In [40]:
def my_neo4j_betweenness(top_n=20):
    """
    Compute betweenness centrality for the subway graph using Neo4j GDS.

    - Re-projects the in-memory graph 'ds_graph' from the current Neo4j store
      (Station nodes, LINK relationships with 'weight').
    - Runs betweenness centrality.
    - Returns a pandas DataFrame of the top N nodes by score.
    """

    # 1. Drop any existing in-memory graph (ignore error if it doesn't exist)
    try:
        drop_query = "CALL gds.graph.drop('ds_graph', false) YIELD graphName"
        session.run(drop_query)
    except Exception:
        # If the graph doesn't exist yet, this may error; we can safely ignore.
        pass

    # 2. Project the graph into GDS memory (same as in my_neo4j_shortest_path)
    project_query = """
    CALL gds.graph.project(
      'ds_graph',
      'Station',
      'LINK',
      { relationshipProperties: 'weight' }
    )
    """
    session.run(project_query)

    # 3. Run betweenness centrality
    #    (no 'normalized' key because this GDS version doesn't support it)
    betweenness_query = """
    CALL gds.betweenness.stream(
      'ds_graph',
      {
        relationshipWeightProperty: 'weight'
      }
    )
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS node_name, score
    ORDER BY score DESC
    LIMIT $top_n
    """

    df = my_neo4j_run_query_pandas(betweenness_query, top_n=top_n)

    print("-------------------------")
    print(" Top", top_n, "nodes by betweenness centrality")
    print("-------------------------")
    print(df)

    return df


In [76]:
# Make sure graph is baseline (no penalties) before running
betweenness_baseline = my_neo4j_betweenness(top_n=20)

-------------------------
 Top 20 nodes by betweenness centrality
-------------------------
                              node_name          score
0               D 59 St-Columbus Circle  284697.500000
1                                D 7 Av  238448.333333
2                              D 125 St  206864.500000
3    5 Franklin Av-Medgar Evers College  192112.333333
4                       Q Prospect Park  179690.100000
5               C 59 St-Columbus Circle  172794.000000
6                                Q 7 Av  172677.933333
7    3 Franklin Av-Medgar Evers College  169194.333333
8   5 President St-Medgar Evers College  166587.000000
9                         5 Sterling St  165823.500000
10                        3 Nostrand Av  163652.333333
11                  A Broadway Junction  162020.250000
12                        5 Winthrop St  159334.500000
13                          5 Church Av  158797.166667
14                              C 50 St  154370.000000
15                        A 

In [83]:
# Make sure graph is penalized 20% before running
betweenness_120 = my_neo4j_betweenness(top_n=20)

-------------------------
 Top 20 nodes by betweenness centrality
-------------------------
                              node_name          score
0    5 Franklin Av-Medgar Evers College  211786.333333
1   5 President St-Medgar Evers College  210645.000000
2                         5 Sterling St  209383.500000
3                         5 Winthrop St  202749.500000
4                           5 Church Av  202138.500000
5    4 Franklin Av-Medgar Evers College  201556.333333
6              4 Crown Heights-Utica Av  198476.500000
7              3 Crown Heights-Utica Av  197096.500000
8               3 Sutter Ave-Rutland Rd  196423.000000
9                         3 Saratoga Av  194231.000000
10                        3 Rockaway Av  192944.500000
11   3 Franklin Av-Medgar Evers College  188560.333333
12                      Q Prospect Park  185044.500000
13                               D 7 Av  179484.333333
14              D 59 St-Columbus Circle  179382.500000
15                          

In [97]:
# Make sure graph is reset then penalized 40% before running
betweenness_140 = my_neo4j_betweenness(top_n=20)

-------------------------
 Top 20 nodes by betweenness centrality
-------------------------
                              node_name          score
0    5 Franklin Av-Medgar Evers College  227550.333333
1   5 President St-Medgar Evers College  227543.000000
2                         5 Sterling St  226867.500000
3                         5 Winthrop St  221090.500000
4                           5 Church Av  220425.666667
5                       Q Prospect Park  215746.500000
6               D 59 St-Columbus Circle  211898.500000
7                                D 7 Av  211834.333333
8    3 Franklin Av-Medgar Evers College  205762.333333
9                           Q Church Av  189855.666667
10                        Q Parkside Av  189515.000000
11                               Q 7 Av  186872.333333
12           N Atlantic Av–Barclays Ctr  160398.333333
13           2 Atlantic Av–Barclays Ctr  143360.333333
14                              1 50 St  143162.500000
15                     1 Tim

# Community Detection (Louvain)

Again, we use Neo4j Graph Data Science (GDS) to detect communities in the subway graph using Louvain algorithm.

This identifies groups of stations that are more tightly connected to each other than to the rest of the network based on the weight property of the links which represents traffic in this case. These communities represent natural clusters or regions of the subway system where travel is denser and more internally connected. Understanding these clusters helps reveal which major zones or transfer groups could be most affected by an increase in traffic such as free fares, which can vary from just typical busier lines or stations.

In [55]:
def my_neo4j_louvain():
    """
    Run Louvain community detection on the subway graph using Neo4j GDS.

    - Re-projects the in-memory graph 'ds_graph'
      (Station nodes, LINK relationships with 'weight').
    - Runs Louvain with intermediate communities.
    - Returns a pandas DataFrame (not top-N limited unless you add a filter).
    """

    # 1. Drop any existing in-memory graph (ignore error if it doesn't exist)
    try:
        drop_query = "CALL gds.graph.drop('ds_graph', false) YIELD graphName"
        session.run(drop_query)
    except Exception:
        pass

    # 2. Project the graph into GDS memory (same as in my_neo4j_shortest_path/my_neo4j_betweenness)
    project_query = """
    CALL gds.graph.project(
      'ds_graph',
      'Station',
      'LINK',
      { relationshipProperties: 'weight' }
    )
    """
    session.run(project_query)

    # 3. Louvain community detection
    louvain_query = """
    CALL gds.louvain.stream(
      'ds_graph',
      {
        includeIntermediateCommunities: true
      }
    )
    YIELD nodeId, communityId, intermediateCommunityIds
    RETURN
      gds.util.asNode(nodeId).name AS name,
      communityId AS community,
      intermediateCommunityIds AS intermediate_community
    ORDER BY community, name ASC
    """

    df = my_neo4j_run_query_pandas(louvain_query)

    return df


In [52]:
# Make sure graph is baseline (no penalties) before running
louvain_baseline = my_neo4j_louvain()
louvain_baseline.head(50)

,name,community,intermediate_community
0,1 125 St,35,"[23, 35, 35, 35, 35, 35]"
1,1 137 St–City College,35,"[31, 35, 35, 35, 35, 35]"
2,1 145 St,35,"[35, 35, 35, 35, 35, 35]"
3,3 116 St,35,"[15, 35, 35, 35, 35, 35]"
4,3 125 St,35,"[23, 35, 35, 35, 35, 35]"
5,3 135 St,35,"[27, 35, 35, 35, 35, 35]"
6,3 145 St,35,"[35, 35, 35, 35, 35, 35]"
7,3 Harlem-148 St,35,"[483, 35, 35, 35, 35, 35]"
8,4 125 St,35,"[23, 35, 35, 35, 35, 35]"
9,4 138 St-Grand Concourse,35,"[33, 37, 35, 35, 35, 35]"


In [70]:
# Make sure graph is penalized 20% before running
louvain_120 = my_neo4j_louvain()
louvain_120.head(50)

,name,community,intermediate_community
0,1 Canal St,125,"[125, 125, 125, 125, 125, 125]"
1,1 Franklin St,125,"[225, 125, 125, 125, 125, 125]"
2,6 Canal St,125,"[125, 125, 125, 125, 125, 125]"
3,6 Spring St,125,"[431, 125, 125, 125, 125, 125]"
4,A Canal St,125,"[125, 125, 125, 125, 125, 125]"
5,C Canal St,125,"[125, 125, 125, 125, 125, 125]"
6,C Spring St,125,"[431, 125, 125, 125, 125, 125]"
7,E Canal St,125,"[125, 125, 125, 125, 125, 125]"
8,E Spring St,125,"[431, 125, 125, 125, 125, 125]"
9,E World Trade Center,125,"[491, 125, 125, 125, 125, 125]"


In [56]:
# Make sure graph is reset then penalized 40% before running
louvain_140 = my_neo4j_louvain()
louvain_140.head(50)

,name,community,intermediate_community
0,1 Canal St,125,"[125, 125, 125, 125, 125, 125]"
1,1 Franklin St,125,"[225, 125, 125, 125, 125, 125]"
2,6 Canal St,125,"[125, 125, 125, 125, 125, 125]"
3,6 Spring St,125,"[431, 125, 125, 125, 125, 125]"
4,A Canal St,125,"[125, 125, 125, 125, 125, 125]"
5,C Canal St,125,"[125, 125, 125, 125, 125, 125]"
6,C Spring St,125,"[431, 125, 125, 125, 125, 125]"
7,E Canal St,125,"[125, 125, 125, 125, 125, 125]"
8,E Spring St,125,"[431, 125, 125, 125, 125, 125]"
9,E World Trade Center,125,"[491, 125, 125, 125, 125, 125]"
